## CASE — mejor estimador de RR desde la PPG de dedo en personas sanas
**Problema:** la variante del notebook 09 (Mediana 3 ≥10) baja el MAE a 1.6 rpm, pero no sigue los cambios
de RR dentro de cada persona (Δ entre vídeos ≈ 0 frente a +1.5 rpm con la banda).

**Protocolo:**
- Misma división del notebook 09 (15 participantes de desarrollo / 15 de prueba, semilla 2026).
- Métodos definidos de antemano. Los que aprenden se entrenan **solo con desarrollo**; su resultado en
  desarrollo es por validación cruzada por participante (5 pliegues), para compararlos en igualdad.
- Se elige con desarrollo (menor MAE mediano por participante) y se informa en prueba.
- **Además del MAE**, se informa si el método sigue los cambios:
  - r dentro de participante (correlación ventana a ventana con la banda);
  - Δ alta − baja activación por participante y su correlación con el Δ de la banda.
- **CNN-LSTM** (arquitectura sin cambios: entrada 3750×1, Conv1D 16/15/3 → MaxPool 3 → Conv1D 32/9 → MaxPool 3 →
  LSTM 16 → Dense 16 → Dense 1, MSE, Adam 0.001) sobre la PPG cruda de 30 s, entrenada solo con desarrollo,
  con aumento por estiramiento temporal (×0.85 y ×1.15, la RR se escala igual) solo en entrenamiento.
- Métricas: MAE, RMSE, sesgo y límites de acuerdo de Bland-Altman (sesgo ± 1.96 DE).
- **Línea base de control:** predecir siempre la RR media del propio participante (oráculo, con la
  referencia). Tiene buen MAE pero cero seguimiento: un método útil debe superarlo en MAE o en r.

Nota: la mitad de prueba ya se vio una vez (notebook 09) para una sola variante; aquí se vuelve a usar
solo para informar, no para elegir.

In [ ]:
# m-1 — Imports, configuración y rutas
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, decimate, detrend, filtfilt, find_peaks, resample_poly
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = BASE_DIR / "results"
CFG = json.loads((OUT / "config_congelada.json").read_text())
SPLIT = json.loads((OUT / "config_case_correccion.json").read_text())
CASE_DEV = set(SPLIT["case_desarrollo"])
FS = 125
VENTANA_S, PASO_S = CFG["ventana_s"], CFG["paso_s"]
RPM_MAX = CFG["rpm_max"]
N = VENTANA_S * FS
FS_M = 4                                   # Hz, series latido a latido remuestreadas
NFFT = 1024
CASE_DIR = BASE_DIR / "data" / "CASE" / "data" / "interpolated" / "physiological"
ARCHIVOS = sorted(CASE_DIR.glob("sub_*.csv"), key=lambda p: int(p.stem.split("_")[1]))
ACTIVACION = {1: 1, 2: 1, 7: 1, 8: 1, 3: 0, 4: 0, 5: 0, 6: 0}
ESTIRAMIENTOS = [0.85, 1.15]             # aumento de datos de la CNN (solo desarrollo)
CNN_CV_DESARROLLO = True                   # False: no hace validación cruzada en desarrollo (más rápido; la CNN no compite en la elección)
BINS = np.arange(4, 41, 1.0)               # rpm: rejilla del espectro usado por los métodos que aprenden
pd.set_option("display.width", 250)
print(len(ARCHIVOS), "participantes |", len(CASE_DEV), "de desarrollo")

In [ ]:
# m-2 — Funciones
def espectro(x, fs, nfft):
    x = detrend(x) * np.hanning(len(x))
    f = np.fft.rfftfreq(nfft, 1 / fs) * 60
    return f, np.abs(np.fft.rfft(x, nfft)) ** 2

def pico(f, P, lo):
    b = (f >= lo) & (f <= RPM_MAX)
    return f[np.argmax(np.where(b, P, -np.inf))]

def latidos(ppg):
    """Picos y valles; series latido a latido limpias: tiempo, RIIV (media pico-valle), RIAV, RIFV."""
    b, a = butter(2, [0.5, 8], btype="band", fs=FS)
    x = filtfilt(b, a, ppg)
    esc = 1.4826 * np.median(np.abs(x - np.median(x)))
    pk, _ = find_peaks(x, distance=int(0.33 * FS), prominence=0.5 * esc)
    vl = np.array([p0 + np.argmin(x[p0:p1]) for p0, p1 in zip(pk[:-1], pk[1:])], int)
    pk = pk[1:]
    t = pk / FS
    ibi = np.diff(t)
    t, pk, vl = t[1:], pk[1:], vl[1:]
    med = pd.Series(ibi).rolling(11, center=True, min_periods=3).median().to_numpy()
    ok = (ibi >= 0.33) & (ibi <= 1.5) & (np.abs(ibi - med) <= 0.3 * med)
    y = ppg - np.median(ppg)
    return t[ok], {"RIIV": ((y[pk] + y[vl]) / 2)[ok], "RIAV": (y[pk] - y[vl])[ok], "RIFV": ibi[ok]}

def serie_4hz(t, v, t0):
    m = (t >= t0) & (t < t0 + VENTANA_S)
    if m.sum() < 15:
        return None
    return np.interp(t0 + np.arange(VENTANA_S * FS_M) / FS_M, t[m], v[m])

def conteo(x, hp):
    """Conteo de respiraciones en el tiempo (tipo 'count-orig', Schäfer y Kratky / Charlton):
    pasa banda [hp, 0.7] Hz; picos por encima de 0.2 × Q3 de los picos, con un valle < 0 entre picos."""
    b, a = butter(2, [hp, 0.7], btype="band", fs=FS_M)
    x = filtfilt(b, a, x)
    pk, _ = find_peaks(x)
    if len(pk) < 3:
        return np.nan
    pk = pk[x[pk] > 0.2 * np.percentile(x[pk], 75)]
    val = [pk[0]] if len(pk) else []
    for p in pk[1:]:
        if x[val[-1]:p].min() < 0:
            val.append(p)
    if len(val) < 3:
        return np.nan
    rr = 60 / np.median(np.diff(val) / FS_M)
    return rr if 6 <= rr <= RPM_MAX else np.nan

def marcas_banda(rsp, fs=1000):
    x = decimate(decimate(rsp, 10, zero_phase=True), 4, zero_phase=True)
    f = fs / 40
    b, a = butter(2, [0.05, 1.0], btype="band", fs=f)
    x = filtfilt(b, a, x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    pk, _ = find_peaks(x, distance=int(f), prominence=0.25 * iqr)
    return pk / f

def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

In [ ]:
# m-3 — Ventanas: métodos de reglas y espectros para los métodos que aprenden (tarda varios minutos)
filas, ESPEC, XPPG, AUG = [], [], [], []      # AUG: (índice de la ventana original, factor, ventana estirada)
for p in ARCHIVOS:
    d = pd.read_csv(p, usecols=["bvp", "rsp", "video"])
    vid = d["video"].to_numpy(int)
    ppg = resample_poly(d["bvp"].to_numpy(float), FS, 1000)
    rsp125 = resample_poly(d["rsp"].to_numpy(float), FS, 1000)
    marcas = marcas_banda(d["rsp"].to_numpy(float))
    b, a = butter(4, CFG["lp_fft_hz"], btype="low", fs=FS)
    riiv_lp = filtfilt(b, a, ppg)
    b, a = butter(4, CFG["lp_art_hz"], btype="low", fs=FS)
    riiv_art = filtfilt(b, a, ppg)
    n5 = 5 * FS
    tipico = np.median([np.std(riiv_art[i:i + n5]) for i in range(0, len(ppg) - n5 + 1, n5)])
    t_l, mods = latidos(ppg)
    for t0 in np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S):
        v = vid[int(t0 * 1000):int((t0 + VENTANA_S) * 1000)]
        if v.min() != v.max() or v[0] not in ACTIVACION:
            continue
        i0 = int(round(t0 * FS))
        series = {k: serie_4hz(t_l, s, t0) for k, s in mods.items()}
        if any(s is None for s in series.values()):
            continue
        ref = rr_ref(marcas, t0, t0 + VENTANA_S)
        f5, P5 = espectro(decimate(decimate(rsp125[i0:i0 + N], 5, zero_phase=True), 5, zero_phase=True), 5, 2048)
        fila = {"Participante": p.stem, "Conjunto": "desarrollo" if p.stem in CASE_DEV else "prueba",
                "Inicio (s)": t0, "Vídeo": int(v[0]), "Activación": ACTIVACION[int(v[0])],
                "RR banda": ref,
                "Ref fiable": bool(np.isfinite(ref) and 6 <= ref <= RPM_MAX and abs(ref - pico(f5, P5, 6)) <= 2),
                "Excursión": max(np.std(riiv_art[i:i + n5]) for i in range(i0, i0 + N - n5 + 1, n5)) / tipico}
        # Métodos de reglas
        fx, Px = espectro(decimate(decimate(riiv_lp[i0:i0 + N], 5, zero_phase=True), 5, zero_phase=True), 5, 2048)
        fila["A: RIIV FFT ≥6 (original)"] = pico(fx, Px, 6)
        fm, esp = None, {}
        for k, s in series.items():
            fm, esp[k] = espectro(s, FS_M, NFFT)
        for lo in (8, 10):
            e = [pico(fm, esp[k], lo) for k in esp]
            fila[f"B: Mediana 3 picos ≥{lo}"] = np.median(e)
            # Fusión espectral: media de los espectros normalizados (cada uno suma 1 en la banda)
            bnd = (fm >= lo) & (fm <= RPM_MAX)
            Pf = sum(esp[k] / esp[k][bnd].sum() for k in esp)
            fila[f"C: Fusión espectral ≥{lo}"] = pico(fm, Pf, lo)
        for hp in (0.13, 0.17):
            c = [conteo(s, hp) for s in series.values()]
            fila[f"D: Conteo mediana 3, pasa-alto {hp} Hz"] = np.nanmedian(c) if np.isfinite(c).sum() >= 2 else np.nan
        filas.append(fila)
        # Espectros normalizados en rejilla fija (características de los métodos que aprenden)
        ESPEC.append(np.concatenate([np.interp(BINS, fm, esp[k] / esp[k][(fm >= 4) & (fm <= 40)].sum())
                                     for k in esp]))
        # PPG cruda de la ventana para la CNN (normalizada por ventana), y copias estiradas si es desarrollo
        z = lambda w: ((w - w.mean()) / (w.std() + 1e-6)).astype(np.float32)
        XPPG.append(z(ppg[i0:i0 + N]))
        if p.stem in CASE_DEV:
            for s_ in ESTIRAMIENTOS:
                L = int(round(N * s_))
                if i0 + L <= len(ppg):
                    AUG.append((len(filas) - 1, s_, z(resample_poly(ppg[i0:i0 + L], N, L)[:N])))
    print(p.stem, end=" ")
R = pd.DataFrame(filas)
R["fila_original"] = np.arange(len(R))
XS = np.log(np.stack(ESPEC) + 1e-6)
XPPG = np.stack(XPPG)
R["Válida"] = R["Ref fiable"] & (R["Excursión"] <= CFG["umbral_art"])

In [ ]:
# m-4 — Suavizado temporal causal (mediana de las 3 últimas ventanas consecutivas del mismo vídeo)
REGLAS = [c for c in R.columns if c[:2] in ("A:", "B:", "C:", "D:")]
R = R.sort_values(["Participante", "Inicio (s)"])
XS = XS[R.index.to_numpy()]                # mantener los espectros alineados con las filas
XPPG = XPPG[R.index.to_numpy()]
R = R.reset_index(drop=True)
bloque = ((R["Participante"] != R["Participante"].shift()) | (R["Vídeo"] != R["Vídeo"].shift())
          | (R["Inicio (s)"].diff() != PASO_S)).cumsum()
for c in ["B: Mediana 3 picos ≥10", "C: Fusión espectral ≥10", "D: Conteo mediana 3, pasa-alto 0.17 Hz"]:
    R[f"{c} + suavizado"] = R.groupby(bloque)[c].transform(lambda s: s.rolling(3, min_periods=1).median())
REGLAS = [c for c in R.columns if c[:2] in ("A:", "B:", "C:", "D:")]

In [ ]:
# m-5 — Métodos que aprenden (espectros de las 3 modulaciones → RR), entrenados solo con desarrollo
dev = (R["Conjunto"] == "desarrollo") & R["Válida"]
APRENDE = {"E: Ridge sobre espectros": lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 12))),
           "F: Gradient boosting sobre espectros": lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                                                                          random_state=0)}
y = R["RR banda"].to_numpy()
grupos = R["Participante"].to_numpy()
idx_dev = np.where(dev)[0]
for nombre, crear in APRENDE.items():
    pred = np.full(len(R), np.nan)
    for tr, te in GroupKFold(min(5, len(set(grupos[idx_dev])))).split(idx_dev, groups=grupos[idx_dev]):      # desarrollo: validación cruzada
        pred[idx_dev[te]] = crear().fit(XS[idx_dev[tr]], y[idx_dev[tr]]).predict(XS[idx_dev[te]])
    resto = R["Conjunto"] == "prueba"
    pred[resto.to_numpy()] = crear().fit(XS[idx_dev], y[idx_dev]).predict(XS[resto.to_numpy()])
    R[nombre] = pred

In [ ]:
# m-5b — CNN-LSTM (tu arquitectura) entrenada solo con desarrollo
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAY_TF = True
except ImportError:
    HAY_TF = False
    print("TensorFlow no está instalado: se omite la CNN-LSTM.")

def crear_modelo():
    m = keras.Sequential([
        layers.Input(shape=(N, 1)),
        layers.Conv1D(16, 15, strides=3, padding="same", activation="relu"),
        layers.MaxPooling1D(3),
        layers.Conv1D(32, 9, padding="same", activation="relu"),
        layers.MaxPooling1D(3),
        layers.LSTM(16),
        layers.Dense(16, activation="relu"),
        layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse", metrics=["mae"])
    return m

pos = {f: i for i, f in enumerate(R["fila_original"])}       # fila original -> fila ordenada
AUG_I = np.array([pos[a[0]] for a in AUG], int)
AUG_S = np.array([a[1] for a in AUG])
AUG_X = np.stack([a[2] for a in AUG]) if AUG else np.empty((0, N), np.float32)

def entrenar_cnn(idx_tr, semilla):
    """Entrena con las ventanas idx_tr (+ sus copias estiradas válidas); 2 participantes para early stopping."""
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(semilla)
    parts = np.unique(grupos[idx_tr])
    val_p = np.random.default_rng(semilla).choice(parts, min(2, max(1, len(parts) // 4)), replace=False)
    va = idx_tr[np.isin(grupos[idx_tr], val_p)]
    tr = idx_tr[~np.isin(grupos[idx_tr], val_p)] if len(parts) > 1 else idx_tr
    a = np.isin(AUG_I, tr) & (AUG_S * y[AUG_I] >= 6) & (AUG_S * y[AUG_I] <= RPM_MAX)
    Xtr = np.concatenate([XPPG[tr], AUG_X[a]])[:, :, None]
    ytr = np.concatenate([y[tr], AUG_S[a] * y[AUG_I[a]]]) / 60
    m = crear_modelo()
    m.fit(Xtr, ytr, validation_data=(XPPG[va][:, :, None], y[va] / 60), epochs=60, batch_size=32, verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    print(f"  CNN: {len(tr)} ventanas + {a.sum()} estiradas, validación {len(va)}")
    return m

if HAY_TF:
    pred = np.full(len(R), np.nan)
    if CNN_CV_DESARROLLO:
        for k, (tr, te) in enumerate(GroupKFold(min(5, len(set(grupos[idx_dev])))).split(idx_dev, groups=grupos[idx_dev])):
            m = entrenar_cnn(idx_dev[tr], k)
            pred[idx_dev[te]] = m.predict(XPPG[idx_dev[te]][:, :, None], verbose=0).ravel() * 60
    m = entrenar_cnn(idx_dev, 100)
    resto = (R["Conjunto"] == "prueba").to_numpy()
    pred[resto] = m.predict(XPPG[resto][:, :, None], verbose=0).ravel() * 60
    m.save(OUT / "cnn_lstm_case_desarrollo.keras")
    R["G: CNN-LSTM (PPG cruda)"] = pred
    APRENDE["G: CNN-LSTM (PPG cruda)"] = None

METODOS = REGLAS + list(APRENDE)
R.drop(columns="fila_original").to_csv(OUT / "case_rr_mejor_ventanas.csv", index=False)
print(R.groupby("Conjunto")["Válida"].agg(["size", "sum"]))

In [ ]:
# m-6 — Métricas: error y seguimiento de cambios
def metricas(g, col):
    e = (g[col] - g["RR banda"]).abs()
    ok = g[col].notna()
    r = g[ok].groupby("Participante").apply(lambda h: h[col].corr(h["RR banda"]) if h[col].std() > 0 else np.nan)
    dlt = g[ok].groupby(["Participante", "Activación"])[[col, "RR banda"]].mean().unstack()
    d_met = dlt[(col, 1)] - dlt[(col, 0)]
    d_ref = dlt[("RR banda", 1)] - dlt[("RR banda", 0)]
    err = (g[col] - g["RR banda"])[ok]
    sesgo, de = err.mean(), err.std()
    return {"MAE mediano": e.groupby(g["Participante"]).mean().median(), "MAE medio": e.mean(),
            "RMSE": np.sqrt((err ** 2).mean()), "Sesgo": sesgo,
            "LdA inf": sesgo - 1.96 * de, "LdA sup": sesgo + 1.96 * de,
            "% ≤ 2 rpm": 100 * (e <= 2).mean(),
            "Cobertura %": 100 * ok.mean(), "r dentro de participante": r.median(),
            "Δ alta−baja (método)": d_met.median(), "Δ alta−baja (banda)": d_ref.median(),
            "r de los Δ": d_met.corr(d_ref)}

V = R[R["Válida"]].copy()
V["Control: media del participante (oráculo)"] = V.groupby("Participante")["RR banda"].transform("mean")
TODOS = METODOS + ["Control: media del participante (oráculo)"]
TAB = {conj: pd.DataFrame({m: metricas(V[V["Conjunto"] == conj], m) for m in TODOS}).T
       for conj in ["desarrollo", "prueba"]}

print("=== DESARROLLO (para elegir) ===")
print(TAB["desarrollo"].sort_values("MAE mediano").round(2).to_string())
cand = TAB["desarrollo"].drop(index="Control: media del participante (oráculo)")
cand = cand[(cand["Cobertura %"] >= 90) & cand["MAE mediano"].notna()]
ELEGIDO = cand["MAE mediano"].idxmin()
print(f"\nMétodo elegido en desarrollo: {ELEGIDO}")

In [ ]:
# m-7 — PRUEBA (evaluación honesta)
filas = ["A: RIIV FFT ≥6 (original)", "B: Mediana 3 picos ≥10", ELEGIDO] + \
        (["G: CNN-LSTM (PPG cruda)"] if "G: CNN-LSTM (PPG cruda)" in TAB["prueba"].index else []) + \
        ["Control: media del participante (oráculo)"]
print("=== PRUEBA ===")
print(TAB["prueba"].loc[list(dict.fromkeys(filas))].round(2).to_string())
print("\nTodos los métodos en prueba (solo informativo):")
print(TAB["prueba"].sort_values("MAE mediano").round(2).to_string())
(OUT / "config_case_rr_mejor.json").write_text(json.dumps({"metodo": ELEGIDO, "semilla_division": 2026}, indent=2))
pd.concat(TAB, names=["Conjunto", "Método"]).to_csv(OUT / "case_rr_mejor_resumen.csv")